# LLM-driven Feature Engineering for Fraud Detection

Pipeline:
LLM -> Feature Generation -> Feature Computation -> DistilBERT -> Metrics -> Feedback Loop


In [ ]:
# Install dependencies (run once)
!pip install pandas numpy scikit-learn transformers openai tqdm

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

# Load dataset
df = pd.read_csv('transaction.csv')
df.head()

In [ ]:
# Label creation: user-level fraud label
fraud_users = df.groupby('User')['Is Fraud?'].max().reset_index()
fraud_users.columns = ['User', 'User_Fraud_Label']

df = df.merge(fraud_users, on='User')
df.head()

In [ ]:
# OpenAI LLM setup
from openai import OpenAI
client = OpenAI(api_key='YOUR_API_KEY')

def generate_features_via_llm(schema_description):
    prompt = f"""
    Given a transaction dataset schema:
    {schema_description}
    Generate behavioral fraud-detection features.

    Output JSON list:
    {{
      "feature_name": "",
      "description": "",
      "type": "numerical/categorical",
      "computation": "",
      "intuition": ""
    }}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    return response.choices[0].message.content

schema = list(df.columns)
llm_features = generate_features_via_llm(schema)
print(llm_features)

In [ ]:
# Example: manually implement a few LLM-suggested features

# Transactions per user
df['txn_count'] = df.groupby('User')['Amount'].transform('count')

# Average transaction amount per user
df['avg_amount'] = df.groupby('User')['Amount'].transform('mean')

# Transaction variance
df['amount_std'] = df.groupby('User')['Amount'].transform('std')

# Unique merchants count
df['unique_merchants'] = df.groupby('User')['Merchant Name'].transform('nunique')

df.head()

In [ ]:
# Convert features to text for DistilBERT

def row_to_text(row):
    return f"User with avg amount {row['avg_amount']}, txn count {row['txn_count']}, std {row['amount_std']}"

df['text'] = df.apply(row_to_text, axis=1)

df[['text', 'User_Fraud_Label']].head()

In [ ]:
# DistilBERT model
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text', 'User_Fraud_Label']])

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

In [ ]:
# Evaluation
metrics = trainer.evaluate()
print(metrics)

In [ ]:
# Feedback loop (simplified)

threshold = 0.80

if metrics['eval_loss'] > 0.5:
    print("Performance low → regenerate features via LLM")
else:
    print("Good features retained")